# Modern APIs with FastAPI

**FastAPI** is a modern, high-performance web framework for building APIs with Python 3.7+, based on standard Python type hints.

**Key advantages over Flask:**
- Automatic API documentation (Swagger UI at `/docs`)
- Built-in data validation via Pydantic
- Async support out of the box
- Much faster (on par with NodeJS/Go)
- Type safety throughout

Install: `pip install fastapi uvicorn`

Run: `uvicorn main:app --reload`

In [ ]:
try:
    import fastapi
    import pydantic
    print(f'FastAPI: {fastapi.__version__}')
    print(f'Pydantic: {pydantic.__version__}')
except ImportError:
    print('Install: pip install fastapi uvicorn')

## 1. Basic FastAPI App

In [ ]:
basic_app = '''
# main.py
from fastapi import FastAPI

app = FastAPI(title="My API", version="1.0.0")

@app.get("/")
def read_root():
    return {"message": "Hello, World!"}

@app.get("/items/{item_id}")
def read_item(item_id: int, q: str = None):  # types are automatic!
    return {"item_id": item_id, "q": q}

# Run with: uvicorn main:app --reload
# Docs at: http://127.0.0.1:8000/docs
'''

print(basic_app)

## 2. Pydantic Models — Request and Response Validation

In [ ]:
from pydantic import BaseModel, Field, EmailStr, field_validator
from typing import Optional
from datetime import datetime

class UserCreate(BaseModel):
    """Request body model for creating a user."""
    name: str = Field(..., min_length=2, max_length=50)
    email: str = Field(..., description='User email address')
    age: int = Field(..., ge=0, le=150)
    bio: Optional[str] = None
    
    @field_validator('name')
    @classmethod
    def name_must_not_contain_special(cls, v):
        if not v.replace(' ', '').isalpha():
            raise ValueError('Name must contain only letters')
        return v.strip()

class UserResponse(BaseModel):
    """Response model — controls what's sent back."""
    id: int
    name: str
    email: str
    created_at: datetime

# Pydantic automatically validates
try:
    user = UserCreate(name='Alice', email='alice@example.com', age=30)
    print('Valid user:', user.model_dump())
except Exception as e:
    print(f'Validation error: {e}')

# This will fail validation
try:
    invalid = UserCreate(name='A', email='not-an-email', age=200)
except Exception as e:
    print(f'Expected validation error: {e}')

## 3. Path, Query, and Body Parameters

In [ ]:
params_example = '''
from fastapi import FastAPI, Path, Query, Body, HTTPException
from pydantic import BaseModel
from typing import Optional

app = FastAPI()

class Item(BaseModel):
    name: str
    price: float
    in_stock: bool = True

@app.get("/items/{item_id}")
def get_item(
    item_id: int = Path(..., gt=0, description="Item ID must be positive"),
    q: Optional[str] = Query(None, max_length=50),
):
    """Get item by ID with optional search query."""
    return {"item_id": item_id, "q": q}

@app.post("/items/", response_model=Item, status_code=201)
def create_item(item: Item):
    """Create a new item. Body is automatically validated."""
    return item

@app.put("/items/{item_id}")
def update_item(item_id: int, item: Item):
    return {"item_id": item_id, **item.dict()}
'''

print(params_example)

## 4. CRUD API with FastAPI (Testable in Notebook)

In [ ]:
from fastapi import FastAPI, HTTPException, status
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
from typing import Optional
from datetime import datetime

app = FastAPI(title='Bookstore API', version='1.0.0')

# Models
class BookCreate(BaseModel):
    title: str = Field(..., min_length=1)
    author: str = Field(..., min_length=1)
    price: float = Field(..., gt=0)
    genre: Optional[str] = None

class Book(BookCreate):
    id: int
    created_at: datetime

# In-memory store
db: dict[int, Book] = {}
counter = [1]

@app.get('/books', response_model=list[Book])
def list_books(genre: Optional[str] = None):
    books = list(db.values())
    if genre:
        books = [b for b in books if b.genre == genre]
    return books

@app.get('/books/{book_id}', response_model=Book)
def get_book(book_id: int):
    if book_id not in db:
        raise HTTPException(status_code=404, detail=f'Book {book_id} not found')
    return db[book_id]

@app.post('/books', response_model=Book, status_code=status.HTTP_201_CREATED)
def create_book(book: BookCreate):
    new_book = Book(id=counter[0], created_at=datetime.now(), **book.model_dump())
    db[counter[0]] = new_book
    counter[0] += 1
    return new_book

@app.put('/books/{book_id}', response_model=Book)
def update_book(book_id: int, book: BookCreate):
    if book_id not in db:
        raise HTTPException(status_code=404, detail='Not found')
    updated = Book(id=book_id, created_at=db[book_id].created_at, **book.model_dump())
    db[book_id] = updated
    return updated

@app.delete('/books/{book_id}', status_code=status.HTTP_204_NO_CONTENT)
def delete_book(book_id: int):
    if book_id not in db:
        raise HTTPException(status_code=404, detail='Not found')
    del db[book_id]


# Test with TestClient
client = TestClient(app)

print('=== Bookstore API Tests ===')

# Create books
r = client.post('/books', json={'title': 'Clean Code', 'author': 'Robert Martin', 'price': 35.99, 'genre': 'Programming'})
book1 = r.json()
print(f'Created: [{book1["id"]}] {book1["title"]} — £{book1["price"]}  (status: {r.status_code})')

r = client.post('/books', json={'title': 'Dune', 'author': 'Frank Herbert', 'price': 12.99, 'genre': 'Sci-Fi'})
book2 = r.json()
print(f'Created: [{book2["id"]}] {book2["title"]} — £{book2["price"]}')

# List all
r = client.get('/books')
print(f'\nAll books ({len(r.json())}):')
for b in r.json():
    print(f'  [{b["id"]}] {b["title"]} by {b["author"]}')

# Filter by genre
r = client.get('/books?genre=Programming')
print(f'\nProgramming books: {[b["title"] for b in r.json()]}')

# Update
r = client.put('/books/1', json={'title': 'Clean Code (2nd Ed)', 'author': 'Robert Martin', 'price': 39.99, 'genre': 'Programming'})
print(f'\nUpdated: {r.json()["title"]} at £{r.json()["price"]}')

# Delete
r = client.delete('/books/2')
print(f'Delete book 2: {r.status_code}')  # 204

# 404
r = client.get('/books/99')
print(f'Get non-existent: {r.status_code} — {r.json()["detail"]}')

## 5. Async Endpoints

In [ ]:
async_code = '''
from fastapi import FastAPI
import asyncio

app = FastAPI()

@app.get("/async-data")
async def get_async_data():
    """Async endpoint — can use await."""
    await asyncio.sleep(0.1)  # non-blocking I/O
    return {"data": "fetched asynchronously"}

@app.get("/sync-data")
def get_sync_data():
    """Sync endpoint — FastAPI runs it in a thread pool."""
    import time
    time.sleep(0.1)  # blocking, but OK for sync endpoints
    return {"data": "fetched synchronously"}
'''

print('Async vs sync endpoint pattern:')
print(async_code)

## 6. Dependency Injection

In [ ]:
di_code = '''
from fastapi import FastAPI, Depends, HTTPException, Header

app = FastAPI()

# Simple dependency
def get_current_user(authorization: str = Header(...)):
    # In real app: validate JWT token
    if authorization != "Bearer valid-token":
        raise HTTPException(status_code=401, detail="Invalid token")
    return {"user_id": 1, "username": "alice"}

# Use dependency in route
@app.get("/protected")
def protected_route(current_user: dict = Depends(get_current_user)):
    return {"message": f"Hello {current_user[\'username\']}!"}

# Paginator dependency
def pagination(skip: int = 0, limit: int = 10):
    return {"skip": skip, "limit": limit}

@app.get("/items")
def list_items(paging: dict = Depends(pagination)):
    items = list(range(100))
    return items[paging["skip"]: paging["skip"] + paging["limit"]]
'''

print(di_code)